# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their unique `@id`. We will review metadata, load record sets, process the data, and visualize important clinical features.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

**Note:** All references to record sets, fields, and columns are made using the `@id` values from the metadata for consistency and reproducibility.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Dataset Description:", metadata.description)
print("Published Date:", getattr(metadata, 'datePublished', 'N/A'))
print("Citation:", getattr(metadata, 'citeAs', 'N/A'))
print("Keywords:", getattr(metadata, 'keywords', 'N/A'))

## 2. Data Overview
Review available record sets and their `@id`s, as well as the fields and columns within each record set. Every entity is referenced by its `@id`.

In [ ]:
# List all available record sets, their IDs, and contained fields/columns
record_sets = dataset.record_sets
print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- RecordSet Name: {getattr(rs, 'name', 'N/A')} (ID: {rs['@id']})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {getattr(field, 'name', 'N/A')} (@id: {field['@id']}, dataType: {getattr(field, 'dataType', 'unknown')})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {getattr(col, 'name', str(col['@id']))} (@id: {col['@id']})")

## 3. Data Extraction
Extract data from each record set into a DataFrame for analysis.

Record sets, fields, and columns are referenced by their `@id` as presented in the previous overview. This section demonstrates loading all tabular record sets into pandas DataFrames, using their `@id`.

In [ ]:
# Collect recordSet @id values from the metadata
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load each record set into a pandas DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
    print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    print(f"Preview for {record_set_id}:")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps to one of the record sets. We demonstrate filtering for the numeric field 'age at diagnosis' referenced by its `@id`, normalization, and grouping by clinical feature. Remember to use the column `@id`s, not their display names.

In [ ]:
# Example EDA: Assume Table 1 has clinical features, including 'age at diagnosis', 'height', etc.
# First, locate the correct Table 1 record set by @id, then identify column IDs for numeric and grouping fields

# Let's assume Table 1's @id and relevant column @id values:
table1_record_set_id = record_set_ids[0] # Replace with actual @id if known

# Examine columns for Table 1
print("Table 1 columns:", dataframes[table1_record_set_id].columns.tolist())

# Assign @id of numeric and grouping fields, replace with actual values from exploration
# Demo usage: Let's say 'age at diagnosis' field has @id 'cr:ageAtDiagnosis', and 'oligomenorrhea' is 'cr:oligomenorrhea'
numeric_field_id = 'cr:ageAtDiagnosis'  # Replace with true @id of age column
group_field_id = 'cr:oligomenorrhea'    # Replace with true @id of group column

# Filtering, normalization, grouping
if numeric_field_id in dataframes[table1_record_set_id].columns:
    threshold = 25
    filtered_df = dataframes[table1_record_set_id][dataframes[table1_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize clinical feature distributions. Using the referenced numeric field (e.g., 'age at diagnosis' with its `@id`), plot the values using matplotlib. You can adjust the visualization to other fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for 'age at diagnosis' field
if numeric_field_id in dataframes[table1_record_set_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[table1_record_set_id][numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (age at diagnosis)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

## 6. Conclusion
This notebook demonstrated:
- How to load the FAIR^2 clinical dataset with `mlcroissant`
- Access metadata and preview available record sets and fields via their `@id`
- Extract all tabular data for analysis and reference columns using Croissant `@id`
- Run basic exploratory analysis, normalization, grouping, filtering, and visualize key numeric attributes

For further analysis, adapt field and record set references according to additional metadata insights. Always reference entities by their `@id` for robustness.